In [11]:
import sys
!{sys.executable} -m pip install -q ultralytics timm opencv-python torch torchvision

In [12]:
import cv2
import numpy as np
import torch
from ultralytics import YOLO
from collections import Counter

CLASSES = ['balaleet', 'egg_tomato', 'fish', 'gaimat', 'halwa', 'karak', 'liver', 'ma3krona', 'nakhaj', 'samboosa', 'tikka']

det = YOLO("models/best.pt")
print("detector ready:", det.task, "| classes:", list(det.names.values()))

detector ready: detect | classes: ['balaleet', 'egg_tomato', 'fish', 'gaimat', 'halwa', 'karak', 'liver', 'ma3krona', 'nakhaj', 'samboosa', 'tikka']


In [20]:
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Could not open camera 0")

w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
fps = fps or 30
video_writer = cv2.VideoWriter("food_detection_webcam.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

print(f"capturing {w}x{h} @ {fps}fps -> food_detection_webcam.avi (press q to quit)")

while cap.isOpened():
    success, im0 = cap.read()
    if not success:
        print("Video frame is empty or processing is complete.")
        break

    results = det.predict(im0, imgsz=640, conf=0.25, verbose=False)
    frame = results[0].plot()

    video_writer.write(frame)
    cv2.imshow("Bahrain Food Detector", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
video_writer.release()
cv2.destroyAllWindows()
print("saved food_detection_webcam.avi")

capturing 640x480 @ 30fps -> food_detection_webcam.avi (press q to quit)
saved food_detection_webcam.avi


In [14]:
import timm
import torch.nn as nn
from torchvision import transforms

ckpt = torch.load("models/big_model.pt", map_location="cpu")
clf = timm.create_model("vit_base_patch16_224", pretrained=False, num_classes=len(CLASSES))
clf.load_state_dict(ckpt["state_dict"])
clf.eval()
print(f"classifier ready | val_acc: {ckpt['val_acc']:.3f}")

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),   
])

classifier ready | val_acc: 0.985


In [17]:
def classify_crop(bgr):
    x = tf(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)).unsqueeze(0)
    with torch.no_grad():
        return CLASSES[clf(x).argmax(1).item()]

def predict_two_stage(frame, conf=0.25):
    out = frame.copy()
    for r in det.predict(frame, imgsz=640, conf=conf, verbose=False)[0].boxes:
        x1, y1, x2, y2 = map(int, r.xyxy[0].tolist())
        name = classify_crop(frame[max(0, y1):y2, max(0, x1):x2])
        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 255, 0), 2)
        label = f"{name} {r.conf.item():.2f}"
        cv2.putText(out, label, (x1, max(0, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    return out

In [21]:
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Could not open camera 0")

w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
fps = fps or 30
video_writer = cv2.VideoWriter("food_two_stage.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

print("two-stage mode | press q to quit")
while cap.isOpened():
    success, im0 = cap.read()
    if not success:
        print("Video frame is empty or processing is complete.")
        break

    frame = predict_two_stage(im0)
    video_writer.write(frame)
    cv2.imshow("Two-Stage Food Detector", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
video_writer.release()
cv2.destroyAllWindows()
print("saved food_two_stage.avi")

two-stage mode | press q to quit
saved food_two_stage.avi
